# Shape and mock-stage exploration

Retained original shape analysis. Raw BST summaries cover regular Pokémon and all rarities; mock-stage projection centres regular Pokémon by stage. These associations do not establish biological mechanisms. Existing datasets remain read only.

In [1]:
# Resolve project root before package imports. Works from root or notebooks/.
import os
import sys
from pathlib import Path

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "pyproject.toml").is_file() and (p / "pokeml").is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Launch this notebook inside the ml_pokemon project")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if sys.version_info[:2] != (3, 14):
    raise RuntimeError("Select project Python 3.14.8 kernel (.pokemon)")
print(f"Project: {PROJECT_ROOT} | Python: {sys.version.split()[0]} | Kernel: {sys.executable}")


Project: /Users/pano/Library/Mobile Documents/com~apple~CloudDocs/coding/portfolio/ml_pokemon | Python: 3.14.8 | Kernel: /Users/pano/venv/pokemon/.pokemon/bin/python


In [2]:
import pandas as pd
import numpy as np
from IPython.display import display
from pokeml.eda.snapshot import load_analysis_snapshot

# Load the minimal_pkdx
df = load_analysis_snapshot()

In [3]:
display(df.describe())

,total_stats,height,weight,id,species_id,evolution_chain_id,generation
count,1082.000000,1082.000000,1082.000000,1082.000000,1082.000000,1082.000000,1082.000000
mean,428.256007,1.223752,68.085305,1021.939926,498.195933,245.861368,4.898336
std,112.575410,1.278376,123.187702,2178.370086,299.497838,160.671231,2.650959
min,175.000000,0.100000,0.100000,1.000000,1.000000,1.000000,1.000000
25%,325.000000,0.500000,8.925000,271.250000,231.250000,97.250000,3.000000
50%,450.000000,1.000000,28.900000,541.500000,497.500000,244.500000,5.000000
75%,508.000000,1.500000,70.375000,811.750000,754.750000,379.000000,7.000000
max,720.000000,20.000000,999.900000,10253.000000,1025.000000,549.000000,9.000000


In [4]:
def project_to_mock_stage(df):
    work = df.copy()

    regular_mask = work['rarity'].eq('regular')
    legendary_mask = work['rarity'].eq('legendary')

    # Stage medians for regular Pokémon
    stage_medians = work.loc[regular_mask].groupby('stage')['total_stats'].median()

    # Shared reference stage for regular Pokémon
    reference_regular = work.loc[regular_mask, 'total_stats'].median()

    # Legendary reference can be separate
    reference_legendary = work.loc[legendary_mask, 'total_stats'].median()

    work['stage_median'] = np.nan
    work['bst_projected_mock_stage'] = np.nan

    # Regular: project to the common mock stage
    work.loc[regular_mask, 'stage_median'] = work.loc[regular_mask, 'stage'].map(stage_medians)
    work.loc[regular_mask, 'bst_projected_mock_stage'] = (
        work.loc[regular_mask, 'total_stats']
        - work.loc[regular_mask, 'stage_median']
        + reference_regular
    )

    # Legendary: project to its own common baseline, if you want it included
    work.loc[legendary_mask, 'stage_median'] = reference_legendary
    work.loc[legendary_mask, 'bst_projected_mock_stage'] = (
        work.loc[legendary_mask, 'total_stats']
        - reference_legendary
        + reference_legendary
    )

    return work, stage_medians, reference_regular, reference_legendary

project_df= project_to_mock_stage(df)[0]
project_df

,name,type_1,type_2,rarity,stage,shape,color,total_stats,height,weight,id,species_id,evolution_chain_id,generation,stage_median,bst_projected_mock_stage
0,bulbasaur,grass,poison,regular,s1c3,quadruped,green,318,0.7,6.9,1,1,1,1,300.0,443.0
1,ivysaur,grass,poison,regular,s2c3,quadruped,green,405,1.0,13.0,2,2,1,1,405.0,425.0
2,venusaur,grass,poison,regular,s3c3,quadruped,green,525,2.0,100.0,3,3,1,1,523.0,427.0
3,charmander,fire,NaN,regular,s1c3,upright,red,309,0.6,8.5,4,4,2,1,300.0,434.0
4,charmeleon,fire,NaN,regular,s2c3,upright,red,405,1.1,19.0,5,5,2,1,405.0,425.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077,decidueye_hisui,grass,fighting,regular,s3c3,wings,brown,530,1.6,37.0,10244,724,374,8,523.0,432.0
1078,dialga_origin,steel,dragon,legendary,single,quadruped,white,680,7.0,848.7,10245,483,244,8,590.0,680.0
1079,palkia_origin,water,dragon,legendary,single,upright,purple,680,6.3,659.0,10246,484,245,8,590.0,680.0
1080,wooper_paldea,poison,ground,regular,s1c2,legs,blue,210,0.4,11.0,10253,194,96,9,308.0,327.0


In [5]:
project_mask = project_df['rarity'].eq('regular')
project_regular = project_df.loc[project_mask]


display(project_regular.groupby("shape")["bst_projected_mock_stage"].agg(["count", f"mean", "median", "std"]))

,count,mean,median,std
shape,,,,
armor,47,398.872340,406.0,50.673558
arms,40,427.000000,427.5,18.632753
ball,49,406.428571,418.0,61.859720
blob,43,403.697674,418.0,54.929190
bug-wings,22,401.409091,398.0,84.190068
fish,47,415.170213,422.0,62.348013
heads,22,431.545455,428.0,40.036563
humanoid,137,416.226277,417.0,47.959081
legs,30,418.033333,417.0,58.245310


In [6]:

regular_mask = df['rarity'].eq('regular')
regular_df =df.loc[regular_mask]
display(regular_df.groupby("shape")["total_stats"].agg(["count", f"mean", "median", "std"]))

,count,mean,median,std
shape,,,,
armor,47,347.744681,330.0,101.284461
arms,40,409.500000,407.5,90.699843
ball,49,371.387755,340.0,99.107016
blob,43,389.093023,405.0,102.943076
bug-wings,22,452.818182,455.0,61.789687
fish,47,405.361702,450.0,99.081966
heads,22,435.590909,467.5,99.051049
humanoid,137,424.116788,455.0,101.120201
legs,30,346.000000,310.0,108.732827


In [7]:
display(df.groupby("shape")["total_stats"].agg(["count", f"mean", "median", "std"]))

,count,mean,median,std
shape,,,,
armor,47,347.744681,330.0,101.284461
arms,52,447.692308,480.0,107.134246
ball,53,371.660377,340.0,103.683495
blob,43,389.093023,405.0,102.943076
bug-wings,22,452.818182,455.0,61.789687
fish,49,414.122449,457.0,106.553123
heads,22,435.590909,467.5,99.051049
humanoid,159,446.723270,480.0,110.153513
legs,30,346.000000,310.0,108.732827


In [8]:
# Armor, blob, bug-wings, heads, legs